# CSA PROMPT — XE từ đầu, full Karpathy10epoch
**T4×2, Internet ON; Save Version → Save & Run All.** Gắn3Datasets: ROI134, semantic-banks130 và prompt-full175 (path ở cell đầu). Không API Secret/ảnhJPEG/COCOJSON/checkpoint caption cũ.
Hai notebook độc lập, chạy song song mỗi notebook2GPU. Cùng fresh commonseed42, bank danh từ/động từ CLIP giữ nguyên, global32caption, groupedimage-sharing, Adam(.9,.99), clip1, FP32. Warmup3epoch8e-6→1e-4;10epoch177110updates/5667470captionevents. NhánhCONTROL khóa/bypass adapter; PROMPT dùng object–relation prompt đúng instance/ROI, zero-output ởstep0 rồi học từupdate1. Ảnh thiếu mappedprompt có residual0; không coi là nhãn âm.
Mỗi epoch đánh giá đủ5000VALID với originalbeam5/max20/CIDEr/NLL và lưu captions. CHAIR/BLEU/ROUGE chấm từ captions đã lưu sautrain; chưa có TEST/SCST/rerank ở bước này. Chỉso với nhánhCONTROL cùng lượt; CSA10cũ là tham khảo do encoderdropout-sharing khác.
4DDP guards cho loss/gradient sharing ởclosed/opened +batch32/tail27 và neuralfixtures phải PASS trướctrain. Guard weights bị bỏ; mainworker khởi tạo mới. So `common_init_sha256` của hai output khi review. Không cam kết tăngCIDEr hoặc hội tụ10epochs.
Parentcap10.5h là giới hạn dừng, không phảiETA/quota. Checkpoint latest chứamodel/Adam/cursor/RNG mỗi1000updates vàtrước/sauVALID; best theoVALIDCIDEr/tieearlier. **Giữ ENTIRE output**, không chỉZIP. Nếu dừng, gửireview và mount toàn bộ output vàoRESUME_ROOT cùngnhánh, cùngnotebook/source trên phiên mới; khôngkhởiđộng lại từ đầu tựđộng. Dữ liệu cache không cần tạo lại. No author-localmodelrun.


In [ ]:
import hashlib,json,os,queue,signal,subprocess,sys,tempfile,threading,time,traceback,urllib.request,zipfile,shutil
from pathlib import Path
ARM='PROMPT'
FULL_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-full-roi-karpathy-v134')
BANK_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-semantic-banks-v130')
PROMPT_ROOT=Path('/kaggle/input/datasets/ducanh2403/csa-prompt-full-karpathy-v175')
RESUME_ROOT='' # Fresh run: empty. Same-arm continuation: entire previous output folder.
OUTPUT=Path('/kaggle/working/csa_prompt_scratch_'+ARM+'_xe10')
WALL_HOURS=10.5 # STOPcap within reported15h; not ETA or guaranteed session/quota.
PARENT_DEADLINE=time.monotonic()+WALL_HOURS*3600
for p in (FULL_ROOT,BANK_ROOT,PROMPT_ROOT):
    if not p.is_dir():raise FileNotFoundError('Attach all3Datasets or correct mount: '+str(p))
if RESUME_ROOT and not Path(RESUME_ROOT).is_dir():raise FileNotFoundError('Mount complete same-arm resume folder')
if OUTPUT.exists():raise RuntimeError('Preserve existing output; choose fresh runtime/output path')
if min(shutil.disk_usage('/tmp').free,shutil.disk_usage('/kaggle/working').free)<8*1024**3:raise RuntimeError('Need8GiB free for stage/checkpoints')
OUTPUT.mkdir(parents=True);SOURCE=Path(tempfile.mkdtemp(prefix='csa-prompt-xe10-',dir='/tmp'));STAGE=SOURCE/'stage';LOG=OUTPUT/'notebook.log'
REVISION='8420162b6760af0b6cb2199ba07e8e511b4a2509'
MAPPING={'experiments/csa_paper/run_sharing_benchmark.py': 'run_sharing_benchmark.py', 'experiments/csa_paper/sharing_data.py': 'sharing_data.py', 'experiments/csa_paper/full_cache_data.py': 'full_cache_data.py', 'experiments/csa_paper/model.py': 'model.py', 'experiments/csa_paper/real_bank_data.py': 'real_bank_data.py', 'experiments/csa_paper/pilot_data.py': 'pilot_data.py', 'experiments/csa_paper/package_full_roi_dataset.py': 'package_full_roi_dataset.py', 'experiments/csa_paper/SAVE_FULL_ROI_V134_PACKAGE.json': 'SAVE_FULL_ROI_V134_PACKAGE.json', 'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json': 'SAVE_BANK_V130_PACKAGE.json', 'experiments/csa_paper/REAL_BANK_INPUT_PACKAGE.json': 'REAL_BANK_INPUT_PACKAGE.json', 'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json': 'lemma_correction_review_text/bank_words_corrected.json', 'experiments/csa_paper/DDP_SHARING_BENCHMARK_PROTOCOL.md': 'DDP_SHARING_BENCHMARK_PROTOCOL.md', 'experiments/csa_paper/run_full_xe.py': 'run_full_xe.py', 'experiments/csa_paper/full_xe_data.py': 'full_xe_data.py', 'experiments/csa_paper/full_xe_decode.py': 'full_xe_decode.py', 'experiments/csa_paper/test_full_xe.py': 'test_full_xe.py', 'experiments/csa_paper/FULL_XE_10_PROTOCOL.md': 'FULL_XE_10_PROTOCOL.md', 'experiments/csa_paper/prompt_adapter_data.py': 'prompt_adapter_data.py', 'experiments/csa_paper/prompt_adapter_model.py': 'prompt_adapter_model.py', 'experiments/csa_paper/run_prompt_mapping_audit.py': 'run_prompt_mapping_audit.py', 'experiments/csa_paper/test_prompt_adapter_data.py': 'test_prompt_adapter_data.py', 'experiments/csa_paper/test_prompt_adapter_model.py': 'test_prompt_adapter_model.py', 'experiments/csa_paper/make_prompt_adapter_notebook.py': 'make_prompt_adapter_notebook.py', 'experiments/csa_paper/audit_assets.py': 'audit_assets.py', 'experiments/csa_paper/data.py': 'data.py', 'experiments/csa_paper/prompt_text_cache.py': 'prompt_text_cache.py', 'experiments/csa_paper/test_prompt_text_cache_data.py': 'test_prompt_text_cache_data.py', 'experiments/csa_paper/make_prompt_text_cache_notebook.py': 'make_prompt_text_cache_notebook.py', 'experiments/csa_paper/make_smoke_notebook.py': 'make_smoke_notebook.py', 'experiments/csa_paper/RERANK_TEST_INPUTS.json': 'RERANK_TEST_INPUTS.json', 'experiments/csa_paper/RECOVER_PROMPT_TEXT_167_CELL.py': 'RECOVER_PROMPT_TEXT_167_CELL.py', 'experiments/csa_paper/full_prompt_cache_data.py': 'full_prompt_cache_data.py', 'experiments/csa_paper/run_full_prompt_cache.py': 'run_full_prompt_cache.py', 'experiments/csa_paper/test_full_prompt_cache_data.py': 'test_full_prompt_cache_data.py', 'experiments/csa_paper/make_full_prompt_cache_notebook.py': 'make_full_prompt_cache_notebook.py', 'experiments/csa_paper/PROMPT_SCRATCH_XE_PLAN.md': 'PROMPT_SCRATCH_XE_PLAN.md', 'experiments/csa_paper/recover_full_prompt_mapping.py': 'recover_full_prompt_mapping.py', 'experiments/csa_paper/prompt_scratch_data.py': 'prompt_scratch_data.py', 'experiments/csa_paper/prompt_scratch_model.py': 'prompt_scratch_model.py', 'experiments/csa_paper/prompt_scratch_preflight.py': 'prompt_scratch_preflight.py', 'experiments/csa_paper/test_prompt_scratch_data.py': 'test_prompt_scratch_data.py', 'experiments/csa_paper/test_prompt_scratch_model.py': 'test_prompt_scratch_model.py', 'experiments/csa_paper/run_prompt_scratch.py': 'run_prompt_scratch.py', 'experiments/csa_paper/make_prompt_scratch_notebooks.py': 'make_prompt_scratch_notebooks.py', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': 'FULL_PROMPT_FRESH_V175_RESULT.json', 'captioning/__init__.py': 'captioning/__init__.py', 'captioning/upt_prompts.py': 'captioning/upt_prompts.py', 'captioning/upt_action_text.py': 'captioning/upt_action_text.py', 'captioning/selected_relation_prompts.py': 'captioning/selected_relation_prompts.py', 'captioning/supervised_relations.py': 'captioning/supervised_relations.py'}
HASHES={'experiments/csa_paper/run_sharing_benchmark.py': '51706ae5cbd96fb0484cb0487a05f14f1b34f4332bd9cb57a299d7d3d5a14d18', 'experiments/csa_paper/sharing_data.py': '68a065f5c055a55b41eae0ea554fbb7f8fd1124e8f5422679fcb1d4878f14829', 'experiments/csa_paper/full_cache_data.py': '58cf4d08310fe03e01441af2ddeb12fd3404f59e82259194a32fe98c8b002bc1', 'experiments/csa_paper/model.py': '139edbb1b73caf3dadaf5c960b25542e282f88baf2ce9bdf141de922ceacbc09', 'experiments/csa_paper/real_bank_data.py': '2c791f8dd856198ec1954a2edd1f9e46520d422f20eff07220e462fc2c9bd9a0', 'experiments/csa_paper/pilot_data.py': '0edab33c7ad7cc54c28efdf6007f23fa6dc4547bdda32e5c0d7d888a3369394d', 'experiments/csa_paper/package_full_roi_dataset.py': 'be3e7d0f96add450a5295add6466e19977da90f486e5e726e3ea40a505cceecb', 'experiments/csa_paper/SAVE_FULL_ROI_V134_PACKAGE.json': 'a73636ca75806d6b1d7b4d88f80f3ce1c59b3061eb3e9a5e730097ddd89c505c', 'experiments/csa_paper/SAVE_BANK_V130_PACKAGE.json': '826ba36628d6587860e2d548e950ea953bd4bbb8c0a8dc64393089dbc1e058e1', 'experiments/csa_paper/REAL_BANK_INPUT_PACKAGE.json': 'c60902e04ea15cf90f652e384937bb317e288753f13b8ede6ab5ef89f8d97129', 'experiments/csa_paper/lemma_correction_review_text/bank_words_corrected.json': 'cec76a0229f30361235a4b64e9712ac6bb2e79fc62a9241079a7af3fa95d13ac', 'experiments/csa_paper/DDP_SHARING_BENCHMARK_PROTOCOL.md': '6537e5e44fb8eb1504c4234b87d4e4e87929cf80fed3884f3caa1ef09468b2f1', 'experiments/csa_paper/run_full_xe.py': 'e3a56c95963d64d1f00e4b21aa18eddd0f2a834d58556136996c7a2323e56101', 'experiments/csa_paper/full_xe_data.py': 'a7a5a3b4d70c9d3a357aa464dfe397a6cba8d851e33065299fb98721ddca7135', 'experiments/csa_paper/full_xe_decode.py': 'b8e618559feec6361cfb3b4619899e4968ab71ab67e113ff38b6d3cd99cdc235', 'experiments/csa_paper/test_full_xe.py': 'aee46db054fd18bcf9306d8d8240414d67ca39dccb74d7ab0337e5d9ef7cb1fa', 'experiments/csa_paper/FULL_XE_10_PROTOCOL.md': 'b52d4081cf0657c7c7fba8a00def7efbff928b4979180e08b4e552cc3983a70c', 'experiments/csa_paper/prompt_adapter_data.py': 'd8b8b61a518d9d4e2d413386d08e60963586bf3b17a16e5c96ee3f54eeaf36c0', 'experiments/csa_paper/prompt_adapter_model.py': '83e5fdb2662940ded06e56df3a0fdf512dc1046b2352e563b59f22a6117e245c', 'experiments/csa_paper/run_prompt_mapping_audit.py': 'ea35c2a582e8903d692f6aaa19735075b474fe108ea4c6ddac467a25b545af5e', 'experiments/csa_paper/test_prompt_adapter_data.py': 'a774dab306aec09aadea869953b1cc04e5592edf1e365b53f02e73258be92851', 'experiments/csa_paper/test_prompt_adapter_model.py': '152d7419238d9484c5b16e84fd987f3e9f68a57649f17019e5532860e144554a', 'experiments/csa_paper/make_prompt_adapter_notebook.py': 'b553a7d68efda2c679b2b0b470fa63f11d772782ce112d186d10b4c05ff55b34', 'experiments/csa_paper/audit_assets.py': '5be2fdf1ee094ef8b23a547b16f423ed3603f7c80b2fe5f2d46cb9224cb90959', 'experiments/csa_paper/data.py': '5a5bb451dc20184d3ea597d4dc8415a120583571f4e1a4dac75e29a306213a71', 'experiments/csa_paper/prompt_text_cache.py': '0bd52c07b44d201a631e6383af111c21d7b3ef2e2ed676f5bc13fc53306e0f23', 'experiments/csa_paper/test_prompt_text_cache_data.py': 'e5cfe2b6ea75c29661265b195d5d7377eb0c5d3cbe497807ed3300a95c5a89d5', 'experiments/csa_paper/make_prompt_text_cache_notebook.py': 'f81983cd170632bdd5b787d3ffd1168be9d3018562d072bd9fc28a6450d84e82', 'experiments/csa_paper/make_smoke_notebook.py': '9de13e286efb3ddd29a2eed331219a9e60dcb384fc2f28e062f64dd8dd1c6054', 'experiments/csa_paper/RERANK_TEST_INPUTS.json': '33395ec35a0803ebf711737f0bdba3b6be8c16244f4c292057658e0fa3feaf46', 'experiments/csa_paper/RECOVER_PROMPT_TEXT_167_CELL.py': 'f91972596641c314ff3c7817889e63429cbb3fcc0f20950fb5db101311d8463a', 'experiments/csa_paper/full_prompt_cache_data.py': 'efb21689e1983e8ab22f9d63068cf262f58344498d412056580c8af54c481f16', 'experiments/csa_paper/run_full_prompt_cache.py': '203ac1482b4a6d23c4ba90b6ad6dac1b31ec1c0f83643bda97d741015e9cc228', 'experiments/csa_paper/test_full_prompt_cache_data.py': 'ef3401f488c0183aadde6097a6d05cbd029127e85594b30eecc4219c25257d75', 'experiments/csa_paper/make_full_prompt_cache_notebook.py': 'e8027ff67b094b92ea787e6e275c1c1f1ec095ede8121dcf664c9fd394d10a93', 'experiments/csa_paper/PROMPT_SCRATCH_XE_PLAN.md': '7fd9abb0667c16d3f3bc6627b047dc228409db9474e1f1c4f9a3a77add09f276', 'experiments/csa_paper/recover_full_prompt_mapping.py': 'b034be6fc1a5972b1ec690c00dcb85b39d58d0357a3d8fd7c8a199659091f6c3', 'experiments/csa_paper/prompt_scratch_data.py': '025496a549e12a49104992e1c48f09968ebf4dcb520d649c818df6b39c0a6959', 'experiments/csa_paper/prompt_scratch_model.py': '31db302359b2e6f850e4b1714a93c46555bf512c9b8afd86bbca6fcc8b6ffec9', 'experiments/csa_paper/prompt_scratch_preflight.py': 'a14ef9d2c43b15341ca57d46a14579e3fbfd9694f6338e1061909ba25d420419', 'experiments/csa_paper/test_prompt_scratch_data.py': '261bfc2d08e9e7f5fb85a1434e6ced0f3a30e5a419c009d0150eedb55cca8581', 'experiments/csa_paper/test_prompt_scratch_model.py': '81304de0b027d15e9b5d17d491473a46c4494d24f811fd588e8e9bd3a990ba3d', 'experiments/csa_paper/run_prompt_scratch.py': '2c8efd86b7ac2a887239a2c7b30773556401ff510a8554f40a88a7ac2a75f1e8', 'experiments/csa_paper/make_prompt_scratch_notebooks.py': '74cfe48a9cf805eac1b559b2274b432b6585e2dd2228d70ae7aab1045a1da890', 'experiments/csa_paper/FULL_PROMPT_FRESH_V175_RESULT.json': '452e94530b07b117be6873f004404b558e29e87cb09e97e27a1548d0bfd5f81a', 'captioning/__init__.py': 'ad259f50a32265d298d027a888f801d378c4669313b8dab06e0653978ce30633', 'captioning/upt_prompts.py': '25bb24346d2b2c649ee7091c72d5def060ddf2c290524a35ef3d301d274fb65c', 'captioning/upt_action_text.py': 'ca0d8f1dc75be3ccfd589fcddc962e87a43222751a4d1beb68098cfa213d526c', 'captioning/selected_relation_prompts.py': 'de13dfbd80e4d4b8c08cd3e40407d5134cb1a5108c58f3eebfd4b74e237179d2', 'captioning/supervised_relations.py': '3e29fd60bbf166ae078581fb9ad34da6ce4462ee287f1c4eb9bbda2abd91f984'}
for original,destination in MAPPING.items():
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent cap during source download')
    raw=urllib.request.urlopen('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/'+original,timeout=min(90,remaining)).read()
    if hashlib.sha256(raw).hexdigest()!=HASHES[original]:raise ValueError('Source SHA differs:'+original)
    p=SOURCE/destination;p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(raw)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2),encoding='utf-8')
os.environ['OMP_NUM_THREADS']='1';os.environ['PYTHONPATH']=str(SOURCE)


In [ ]:
def run(args, timeout=1800):
    if time.monotonic()>=PARENT_DEADLINE:raise TimeoutError('Parent wall cap')
    timeout=min(timeout,max(.01,PARENT_DEADLINE-time.monotonic()))
    print('Running:', ' '.join(map(str, args)), flush=True)
    process = subprocess.Popen(list(map(str, args)), cwd=SOURCE, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, errors='replace', bufsize=1, start_new_session=True)
    messages = queue.Queue()
    def reader():
        try:
            for line in process.stdout: messages.put(line)
        finally: messages.put(None)
    thread = threading.Thread(target=reader, daemon=True); thread.start()
    started = heartbeat = time.monotonic(); ended = False
    with LOG.open('a', encoding='utf-8') as log:
        try:
            while not ended or process.poll() is None or not messages.empty():
                try:
                    line = messages.get(timeout=1)
                    if line is None: ended = True
                    else: print(line, end='', flush=True); log.write(line); log.flush()
                except queue.Empty: pass
                now = time.monotonic()
                if now-heartbeat >= 30:
                    print('Prompt XE active;', round(now-started), 'seconds', flush=True); heartbeat=now
                if now-started > timeout: raise TimeoutError('Prompt XE wall cap exceeded')
            if process.wait(): raise subprocess.CalledProcessError(process.returncode, args)
        finally:
            if process.poll() is None:
                os.killpg(process.pid, signal.SIGTERM)
                try: process.wait(timeout=10)
                except subprocess.TimeoutExpired: os.killpg(process.pid, signal.SIGKILL); process.wait()
            thread.join(timeout=5)


In [ ]:
complete=False
common=['--bank-root',BANK_ROOT,'--prompt-root',PROMPT_ROOT,'--stage',STAGE,'--output',OUTPUT,'--arm',ARM]
launch=[sys.executable,'-u','-m','torch.distributed.run','--standalone','--nnodes=1','--nproc_per_node=2','--max_restarts=0']
try:
    run([sys.executable,'-u',SOURCE/'test_prompt_adapter_data.py'],timeout=120)
    run([sys.executable,'-u',SOURCE/'test_prompt_scratch_data.py'],timeout=120)
    # Neural fixtures run here on Kaggle only; no trained weights survive these processes.
    run([sys.executable,'-u',SOURCE/'test_prompt_adapter_model.py'],timeout=180)
    run([sys.executable,'-u',SOURCE/'test_prompt_scratch_model.py'],timeout=180)
    run([sys.executable,'-m','pip','install','-q','pycocoevalcap==1.2'],timeout=300)
    run([sys.executable,'-u',SOURCE/'run_prompt_scratch.py','prepare','--full-root',FULL_ROOT,*common],timeout=1800)
    run([sys.executable,'-u',SOURCE/'run_prompt_scratch.py','metric_preflight',*common],timeout=300)
    run([*launch,SOURCE/'prompt_scratch_preflight.py',*common],timeout=1200)
    guard=json.loads((OUTPUT/'prompt_preflight_result.json').read_text(encoding='utf-8'))
    if not guard['passed'] or guard['cases']!=4 or guard['arm']!=ARM:raise RuntimeError('Missing exact-arm DDP guards')
    remaining=PARENT_DEADLINE-time.monotonic()
    if remaining<600:raise TimeoutError('No training budget after preparation')
    run([*launch,SOURCE/'run_prompt_scratch.py','worker',*common,'--resume-root',RESUME_ROOT,'--wall-seconds',remaining-60],timeout=remaining)
    result=OUTPUT/'result.json'
    complete=result.is_file() and json.loads(result.read_text(encoding='utf-8')).get('status')=='PROMPT_SCRATCH_XE10_COMPLETE'
    if not complete:raise RuntimeError('Stopped before10epochs; preserve ENTIRE output for same-arm resume')
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,arm=ARM,requested_epochs=10,test_used=False)),encoding='utf-8')
    REVIEW=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(REVIEW,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(OUTPUT.rglob('*')):
            if p.is_file() and p.suffix in ('.json','.log','.txt'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',REVIEW,'\nKEEP ENTIRE OUTPUT:',OUTPUT,flush=True)
    for n in ('checkpoint.bin','best.bin'):
        if (OUTPUT/n).is_file():print('KEEP CHECKPOINT:',OUTPUT/n,flush=True)
    print('COMPLETE10' if complete else 'INCOMPLETE; preserve output/logs, no automatic restart.',flush=True)
